![image info](https://raw.githubusercontent.com/albahnsen/MIAD_ML_and_NLP/main/images/banner_1.png)

# Taller: Construcción e implementación de modelos Bagging, Random Forest y XGBoost

En este taller podrán poner en práctica sus conocimientos sobre la construcción e implementación de modelos de Bagging, Random Forest y XGBoost. El taller está constituido por 8 puntos, en los cuales deberan seguir las intrucciones de cada numeral para su desarrollo.

## Datos predicción precio de automóviles

En este taller se usará el conjunto de datos de Car Listings de Kaggle donde cada observación representa el precio de un automóvil teniendo en cuenta distintas variables como año, marca, modelo, entre otras. El objetivo es predecir el precio del automóvil. Para más detalles puede visitar el siguiente enlace: [datos](https://www.kaggle.com/jpayne/852k-used-car-listings).

In [32]:
import warnings
warnings.filterwarnings('ignore')

In [33]:
# Importación de librerías
%matplotlib inline
import pandas as pd
import numpy as np
from sklearn.metrics import mean_squared_error, mean_absolute_error
from sklearn.tree import DecisionTreeRegressor

# Lectura de la información de archivo .csv
data = pd.read_csv('https://raw.githubusercontent.com/albahnsen/MIAD_ML_and_NLP/main/datasets/dataTrain_carListings.zip')

# Preprocesamiento de datos para el taller
data = data.loc[data['Model'].str.contains('Camry')].drop(['Make', 'State'], axis=1)
data = data.join(pd.get_dummies(data['Model'], prefix='M'))
data = data.drop(['Model'], axis=1)

# Visualización dataset
data.head()

,Price,Year,Mileage,M_Camry,M_Camry4dr,M_CamryBase,M_CamryL,M_CamryLE,M_CamrySE,M_CamryXLE
7,21995,2014,6480,False,False,False,True,False,False,False
11,13995,2014,39972,False,False,False,False,True,False,False
167,17941,2016,18989,False,False,False,False,False,True,False
225,12493,2014,51330,False,False,False,True,False,False,False
270,7994,2007,116065,False,True,False,False,False,False,False


In [34]:
# Separación de variables predictoras (X) y variable de interés (y)
y = data['Price']
X = data.drop(['Price'], axis=1)

In [35]:
# Separación de datos en set de entrenamiento y test
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.33, random_state=42)

### Punto 1 - Árbol de decisión manual

En la celda 1 creen un árbol de decisión **manualmente**  que considere los set de entrenamiento y test definidos anteriormente y presenten el RMSE y MAE del modelo en el set de test.

In [36]:
# Celda 1


# Función MSE como criterio en problemas de regresión para hacer cada split del árbol
def mse(y):
    return np.mean((y - y.mean())**2)


# Función para calcular MSE del split
def mse_split(x, y, split):
    left = y[x < split]
    right = y[x >= split]
    # Evitar divisiones inválidas
    if len(left) == 0 or len(right) == 0:
        return np.inf
    mse_left = mse(left)
    mse_right = mse(right)
    mse_total = (len(left)/len(y))*mse_left + (len(right)/len(y))*mse_right
    return mse_total


# Definición de la función best_split para calcular cuál es la mejor variable y punto de corte para hacer la bifurcación del árbol
def best_split(X, y, num_pct=10):
    # Usar solo columnas numéricas (float/int), ignorando booleanas y objetos
    X_num = X.select_dtypes(include=['number'])
    features = range(X_num.shape[1])
    best_split = [0, 0, np.inf]  # j, split, score
    # Para todas las variables numéricas
    for j in features:
        col = X_num.iloc[:, j].astype(float)
        # Evitar columnas constantes (sin varianza)
        if col.nunique() <= 1:
            continue
        splits = np.percentile(col, np.arange(0, 100, 100.0 / (num_pct+1)).tolist())
        splits = np.unique(splits)[1:]
        # Para cada partición
        for split in splits:
            score = mse_split(col, y, split)
            if score  < best_split[2]:
                best_split = [j, split, score]
    return best_split


# Definición de la función tree_grow para hacer un crecimiento recursivo del árbol de regresión
def tree_grow(X, y, level=0, min_gain=1000, max_depth=None, num_pct=10):
    # Si solo es una observación
    if X.shape[0] == 1:
        tree = dict(y_pred=y.iloc[:1].values[0], level=level, split=-1, n_samples=1, score=0)
        return tree
    # Trabajar solo con columnas numéricas
    X_num = X.select_dtypes(include=['number'])
    # Calcular la mejor división
    j, split, score = best_split(X_num, y, num_pct)
    # Predicción = media
    y_pred = y.mean()
    tree = dict(y_pred=y_pred, level=level, split=-1, n_samples=X.shape[0], score=score)
    # Calcular ganancia
    gain = mse(y) - score
    # Revisar el criterio de parada
    if gain < min_gain:
        return tree
    if max_depth is not None:
        if level >= max_depth:
            return tree
    # Continuar creando la partición
    col = X_num.iloc[:, j]
    filter_l = col < split
    X_l, y_l = X.loc[filter_l], y.loc[filter_l]
    X_r, y_r = X.loc[~filter_l], y.loc[~filter_l]
    tree['split'] = [X_num.columns[j], split]  # guardar el nombre de la columna
    # Siguiente iteración para cada partición
    tree['sl'] = tree_grow(X_l, y_l, level + 1, min_gain=min_gain, max_depth=max_depth, num_pct=num_pct)
    tree['sr'] = tree_grow(X_r, y_r, level + 1, min_gain=min_gain, max_depth=max_depth, num_pct=num_pct)
    return tree


# Definición de la función tree_predict para hacer predicciones según las variables 'X' y el árbol 'tree'
def tree_predict(X, tree):
    predicted = np.ones(X.shape[0])
    # Revisar si es el nodo final
    if tree['split'] == -1:
        predicted = predicted * tree['y_pred']
    else:
        col_name, split = tree['split']
        col = X[col_name]
        filter_l = (col < split)
        X_l = X.loc[filter_l]
        X_r = X.loc[~filter_l]
        if X_l.shape[0] == 0:  # Si el nodo izquierdo está vacío solo continua con el derecho
            predicted[~filter_l] = tree_predict(X_r, tree['sr'])
        elif X_r.shape[0] == 0:  #  Si el nodo derecho está vacío solo continua con el izquierdo
            predicted[filter_l] = tree_predict(X_l, tree['sl'])
        else:
            predicted[filter_l] = tree_predict(X_l, tree['sl'])
            predicted[~filter_l] = tree_predict(X_r, tree['sr'])
    return predicted



In [37]:
#Entrenar arbol usando train
tree = tree_grow(X_train, y_train, level=0, min_gain=1000, max_depth=5, num_pct=10)
tree

# Ejecución de función tree_predict usando test
y_pred_test = tree_predict(X_test, tree)
print("Valores predichos:")
print(y_pred_test)

#Cálculo del RMSE y MAE
rmse = np.sqrt(mean_squared_error(y_test, y_pred_test))
mae = mean_absolute_error(y_test, y_pred_test)
print("\nRMSE Árbol manual:", rmse)
print("MAE Árbol manual:", mae)


Valores predichos:
[13496.40909091  7785.453125   15755.06794425 ... 18643.9380531
 12503.56140351 10789.80327869]

RMSE Árbol manual: 1742.1476422077644
MAE Árbol manual: 1293.2123287960212


### Punto 2 - Bagging manual

En la celda 2 creen un modelo bagging **manualmente** con 10 árboles de regresión y comenten sobre el desempeño del modelo.

In [38]:
# Celda 2

#Resetear los indices antes de hacer el bagging para poder realizar las muestras de bootstrap
X_train = X_train.reset_index(drop=True)
y_train = y_train.reset_index(drop=True)

# Se crea un arreglo de 1 a 7100 y se imprime el arreglo y muestreo aleatorio
np.random.seed(123)
nums = np.arange(0, 7100)
print('Arreglo:', nums)
print('Muestreo aleatorio: ', np.random.choice(a=nums, size=7100, replace=True))

# Creación de 10 muestras de bootstrap
n_samples = X_train.shape[0]
n_B = 10
samples = [np.random.choice(a=n_samples, size=n_samples, replace=True) for _ in range(1, n_B +1 )]
samples

# Visualización muestra boostrap #1 para entremiento
print('\nMuestra Boostrap #1:')
X_train.iloc[samples[0], :]

Arreglo: [   0    1    2 ... 7097 7098 7099]
Muestreo aleatorio:  [3582 3454 1346 ... 5863 3492 4254]

Muestra Boostrap #1:


,Year,Mileage,M_Camry,M_Camry4dr,M_CamryBase,M_CamryL,M_CamryLE,M_CamrySE,M_CamryXLE
4964,2012,83553,False,True,False,False,False,False,False
2938,2017,30929,False,False,False,False,False,True,False
3462,2014,44020,False,False,False,True,False,False,False
3447,2017,2008,False,False,False,False,False,True,False
4777,2007,117433,False,True,False,False,False,False,False
...,...,...,...,...,...,...,...,...,...
4391,2006,124735,False,True,False,False,False,False,False
484,2014,57487,False,False,False,False,True,False,False
2521,2011,89766,False,True,False,False,False,False,False
2466,2012,57859,False,False,False,False,False,True,False


In [39]:
# Definición del modelo usando DecisionTreeRegressor de sklearn
treereg = DecisionTreeRegressor(max_depth=None, random_state=123)

# DataFrame para guardar las predicciones de cada árbol
y_pred = pd.DataFrame(index=X_test.index, columns=[list(range(n_B))])

# Entrenamiento de un árbol sobre cada muestra boostrap y predicción sobre los datos de test
for i, sample in enumerate(samples):
    X_boot  = X_train.iloc[sample, :]
    y_boot  = y_train.iloc[sample]
    treereg.fit(X_boot, y_boot)
    y_pred.iloc[:,i] = treereg.predict(X_test)

y_pred

,0,1,2,3,4,5,6,7,8,9
257343,13993.0,13649.0,13649.0,13990.0,13649.0,13993.0,13993.0,13990.0,13993.0,13649.0
326011,5995.0,5995.0,6987.0,5995.0,5995.0,5995.0,6987.0,5995.0,5995.0,5995.0
242354,16995.0,16491.0,15997.0,15997.0,16491.0,17591.0,16995.0,17404.0,16491.0,16491.0
266376,21990.0,22500.0,21990.0,15900.0,21990.0,21990.0,21990.0,15813.0,21990.0,15900.0
396954,16951.0,15988.0,15988.0,15988.0,17900.0,16951.0,16951.0,15988.0,15988.0,15988.0
...,...,...,...,...,...,...,...,...,...,...
144298,14800.0,14800.0,14800.0,14800.0,14681.0,14800.0,14800.0,14800.0,13836.0,14800.0
364521,14995.0,15999.0,17987.0,15999.0,15999.0,15999.0,15999.0,16999.0,14851.0,15999.0
120072,23533.0,20000.0,17700.0,17700.0,23533.0,17700.0,23533.0,23533.0,20000.0,17700.0
99878,12991.0,12989.0,12995.0,12991.0,12991.0,10995.0,12991.0,12991.0,12893.0,12989.0


In [40]:
# Desempeño de cada árbol
for i in range(n_B):
    print('Árbol ', i, 'tiene un RMSE: ', np.sqrt(mean_squared_error(y_test,y_pred.iloc[:,i])))


# Predicciones promedio para cada observación del set de test
y_pred.mean(axis=1)

#Cálculo del RMSE y MAE
np.sqrt(mean_squared_error(y_test, y_pred.mean(axis=1)))

#Cálculo del RMSE y MAE al promediar las predicciones de todos los árboles
rmse = np.sqrt(mean_squared_error(y_test, y_pred.mean(axis=1)))
mae = mean_absolute_error(y_test, y_pred.mean(axis=1))
print("\nRMSE Bagging manual:", rmse)
print("MAE Bagging manual:", mae)

Árbol  0 tiene un RMSE:  2141.9171501683118
Árbol  1 tiene un RMSE:  2125.6987529878875
Árbol  2 tiene un RMSE:  2082.9814991396815
Árbol  3 tiene un RMSE:  2165.747795521846
Árbol  4 tiene un RMSE:  2116.26408903415
Árbol  5 tiene un RMSE:  2153.0973164388097
Árbol  6 tiene un RMSE:  2194.303636378142
Árbol  7 tiene un RMSE:  2142.4593794405823
Árbol  8 tiene un RMSE:  2136.6026071273845
Árbol  9 tiene un RMSE:  2135.7804645213514

RMSE Bagging manual: 1798.8880759942176
MAE Bagging manual: 1341.2787475943035


**Interpretación**

El desempeño de los árboles individuales muestra errores relativamente similares, con valores de RMSE entre aproximadamente 2083 y 2194, lo que indica que cada árbol tiene una capacidad predictiva parecida, aunque con cierta variabilidad debido al muestreo bootstrap. Sin embargo, al aplicar bagging y promediar las predicciones de los 10 árboles, el error disminuye de forma notable, obteniendo un RMSE de 1798.89 y un MAE de 1341.28. Este resultado es considerablemente mejor que el de cualquiera de los árboles individuales, lo que evidencia que el bagging logra reducir la varianza del modelo, ya que la combinación de múltiples árboles mediante promediado permite obtener predicciones más robustas y precisas.

### Punto 3 - Bagging con librería

En la celda 3, con la librería sklearn, entrenen un modelo bagging con 10 árboles de regresión y el parámetro `max_features` del árbol de decisión igual a `log(n_features)` y comenten sobre el desempeño del modelo.

In [41]:
# Celda 3

from sklearn.ensemble import BaggingRegressor
import numpy as np

# Número de features
n_features = X_train.shape[1]

# Modelo Bagging
bagging = BaggingRegressor(
    estimator=DecisionTreeRegressor(random_state=123),
    n_estimators=10,
    max_features=int(np.log(n_features)),  # log(n_features)
    random_state=123
)

# Entrenamiento
bagging.fit(X_train, y_train)

# Predicción
y_pred_bag = bagging.predict(X_test)

# Métricas
rmse = np.sqrt(mean_squared_error(y_test, y_pred_bag))
mae = mean_absolute_error(y_test, y_pred_bag)

print("RMSE Bagging (sklearn):", rmse)
print("MAE Bagging (sklearn):", mae)

RMSE Bagging (sklearn): 2286.433510607307
MAE Bagging (sklearn): 1746.1464691166027


# **Comentarios**
El modelo de Bagging implementado con sklearn obtuvo un RMSE de 2286.43 y un MAE de 1746.15, lo cual representa un desempeño inferior en comparación con el bagging manual previamente construido.

Este resultado se explica principalmente por el uso del parámetro max_features = log(n_features), el cual restringe significativamente el número de variables que cada árbol puede utilizar durante el entrenamiento. Como consecuencia, cada modelo individual pierde capacidad predictiva al trabajar con información limitada, generando un fenómeno de subajuste.

Aunque el bagging busca reducir la varianza mediante el uso de múltiples modelos, en este caso la alta restricción en las variables provoca que los árboles sean demasiado débiles, aumentando el error global del ensamble. Esto evidencia que existe un equilibrio entre diversidad y capacidad predictiva: demasiada aleatoriedad puede deteriorar el desempeño del modelo.

### Punto 4 - Random forest con librería

En la celda 4, usando la librería sklearn entrenen un modelo de Randon Forest para regresión  y comenten sobre el desempeño del modelo.

In [42]:
# Celda 4

from sklearn.ensemble import RandomForestRegressor

# Modelo Random Forest
rf = RandomForestRegressor(
    n_estimators=100,
    max_features='log2',
    random_state=123
)

# Entrenamiento
rf.fit(X_train, y_train)

# Predicción
y_pred_rf = rf.predict(X_test)

# Métricas
rmse = np.sqrt(mean_squared_error(y_test, y_pred_rf))
mae = mean_absolute_error(y_test, y_pred_rf)

print("RMSE Random Forest:", rmse)
print("MAE Random Forest:", mae)

RMSE Random Forest: 1786.4180877009082
MAE Random Forest: 1334.1531191554066


# **Comentarios**
El modelo de Random Forest obtuvo un RMSE de 1786.42 y un MAE de 1334.15, mostrando un mejor desempeño en comparación con el Bagging implementado con sklearn y resultados similares o ligeramente mejores que el bagging manual.

Este mejor desempeño se debe a que Random Forest no solo aplica bagging (muestreo bootstrap), sino que también introduce aleatoriedad en la selección de variables en cada división del árbol. Esto reduce la correlación entre los árboles del ensamble, permitiendo que el modelo capture mejor la variabilidad de los datos y genere predicciones más precisas.

A diferencia del Bagging con restricción fuerte en max_features, Random Forest logra un equilibrio adecuado entre diversidad y capacidad predictiva, evitando el subajuste y mejorando la generalización del modelo.

### Punto 5 - Calibración de parámetros Random forest

En la celda 5, calibren los parámetros max_depth, max_features y n_estimators del modelo de Randon Forest para regresión, comenten sobre el desempeño del modelo y describan cómo cada parámetro afecta el desempeño del modelo.

In [43]:
# Celda 5

from sklearn.model_selection import ParameterGrid

param_grid = {
    'n_estimators': [50, 100],
    'max_depth': [5, 10, None],
    'max_features': ['sqrt', 'log2']
}

resultados_rf = []

for params in ParameterGrid(param_grid):
    rf = RandomForestRegressor(
        n_estimators=params['n_estimators'],
        max_depth=params['max_depth'],
        max_features=params['max_features'],
        random_state=123
,    )
    
    rf.fit(X_train, y_train)
    y_pred = rf.predict(X_test)
    
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    
    resultados_rf.append((params, rmse, y_pred))

# Mejor modelo
mejor_modelo = sorted(resultados_rf, key=lambda x: x[1])[0]
mejores_params_rf = mejor_modelo[0]
mejor_rmse_rf = mejor_modelo[1]
y_pred_rf_best = mejor_modelo[2]

print("Mejor modelo:", mejores_params_rf)
print("RMSE:", mejor_rmse_rf)

Mejor modelo: {'max_depth': 10, 'max_features': 'sqrt', 'n_estimators': 50}
RMSE: 1565.6284908144569


**Comentarios Punto 5**

El modelo de Random Forest calibrado con la configuración max_depth = 10, max_features = 'sqrt' y n_estimators = 50 alcanza un RMSE de aproximadamente 1565.63, mejorando de manera evidente el desempeño frente al modelo base. Este resultado indica que una profundidad moderada y una selección aleatoria de un subconjunto de variables por división permiten capturar relaciones más complejas entre las variables sin caer en un sobreajuste severo, logrando un mejor equilibrio entre sesgo y varianza. Además, el hecho de no requerir un número muy alto de árboles (50) sugiere que el modelo es relativamente eficiente en términos computacionales para el nivel de precisión obtenido.

### Punto 6 - XGBoost con librería

En la celda 6 implementen un modelo XGBoost de regresión con la librería sklearn y comenten sobre el desempeño del modelo.

In [44]:
# Celda 6

from xgboost import XGBRegressor

xgb = XGBRegressor(
    n_estimators=100,
    learning_rate=0.1,
    random_state=123
)

xgb.fit(X_train, y_train)

y_pred_xgb = xgb.predict(X_test)

rmse = np.sqrt(mean_squared_error(y_test, y_pred_xgb))
mae = mean_absolute_error(y_test, y_pred_xgb)

print("RMSE XGBoost:", rmse)
print("MAE XGBoost:", mae)

RMSE XGBoost: 1563.297956245066
MAE XGBoost: 1146.633544921875


**Comentarios Punto 6**




El modelo de XGBoost en su versión base alcanza un RMSE de aproximadamente 1563.30 y un MAE cercano a 1146.63, mostrando un desempeño muy competitivo frente al Random Forest calibrado. Esto refleja la capacidad del enfoque de boosting para corregir iterativamente los errores de los árboles anteriores y concentrarse en las observaciones más difíciles, lo que se traduce en un modelo con mayor poder predictivo. Sin embargo, al no estar aún calibrado, existe margen de mejora adicional a través del ajuste fino de hiperparámetros como la tasa de aprendizaje y la regularización.

### Punto 7 - Calibración de parámetros XGBoost

En la celda 7 calibren los parámetros learning rate, gamma y colsample_bytree del modelo XGBoost para regresión, comenten sobre el desempeño del modelo y describan cómo cada parámetro afecta el desempeño del modelo.

In [45]:
# Celda 7

param_grid_xgb = {
    'learning_rate': [0.01, 0.1],
    'gamma': [0, 1],
    'colsample_bytree': [0.5, 1]
}

resultados_xgb = []

for params in ParameterGrid(param_grid_xgb):
    xgb = XGBRegressor(
        n_estimators=100,
        learning_rate=params['learning_rate'],
        gamma=params['gamma'],
        colsample_bytree=params['colsample_bytree'],
        random_state=123
,    )
    
    xgb.fit(X_train, y_train)
    y_pred = xgb.predict(X_test)
    
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    
    resultados_xgb.append((params, rmse, y_pred))

# Mejor configuración
mejor_xgb = sorted(resultados_xgb, key=lambda x: x[1])[0]
mejores_params_xgb = mejor_xgb[0]
mejor_rmse_xgb = mejor_xgb[1]
y_pred_xgb_best = mejor_xgb[2]

print("Mejor configuración XGBoost:", mejores_params_xgb)
print("RMSE:", mejor_rmse_xgb)

Mejor configuración XGBoost: {'colsample_bytree': 0.5, 'gamma': 0, 'learning_rate': 0.1}
RMSE: 1548.0111433707445


**Comentarios Punto 7**




La configuración calibrada de XGBoost con learning_rate = 0.1, gamma = 0 y colsample_bytree = 0.5 logra un RMSE cercano a 1548.01, convirtiéndose en el modelo con mejor desempeño entre los evaluados. Esta mejora frente a la versión base confirma que el ajuste adecuado de los hiperparámetros permite controlar la complejidad del modelo, introducir regularización y aprovechar de forma más eficiente el muestreo de columnas, reduciendo el error sin sacrificar la capacidad de generalización. En consecuencia, XGBoost calibrado se posiciona como la opción más robusta para la predicción del precio de los automóviles en este conjunto de datos.

### Punto 8 - Comparación y análisis de resultados
En la celda 8 comparen los resultados obtenidos de los diferentes modelos (random forest y XGBoost) y comenten las ventajas del mejor modelo y las desventajas del modelo con el menor desempeño.

In [ ]:
# Celda 8

# Resumen de resultados de Random Forest y XGBoost

resultados = {}

# Random Forest base (Celda 4)
resultados['RandomForest_base'] = {
    'RMSE': float(np.sqrt(mean_squared_error(y_test, y_pred_rf))),
    'MAE': float(mean_absolute_error(y_test, y_pred_rf))
}

# Random Forest calibrado (Celda 5)
resultados['RandomForest_calibrado'] = {
    'RMSE': float(mejor_rmse_rf),
    'MAE': float(mean_absolute_error(y_test, y_pred_rf_best))
}

# XGBoost base (Celda 6)
resultados['XGBoost_base'] = {
    'RMSE': float(np.sqrt(mean_squared_error(y_test, y_pred_xgb))),
    'MAE': float(mean_absolute_error(y_test, y_pred_xgb))
}

# XGBoost calibrado (Celda 7)
resultados['XGBoost_calibrado'] = {
    'RMSE': float(mejor_rmse_xgb),
    'MAE': float(mean_absolute_error(y_test, y_pred_xgb_best))
}

df_resultados = pd.DataFrame(resultados).T
df_resultados

,RMSE,MAE
RandomForest_base,1786.418088,1334.153119
RandomForest_calibrado,1565.628491,1148.922322
XGBoost_base,1563.297956,1146.633545
XGBoost_calibrado,1548.011143,1137.539673


### Punto 8 - Interpretación de resultados y comparación de modelos

Al comparar los resultados, se observa que tanto la calibración de hiperparámetros como el uso de XGBoost mejoran de forma consistente el desempeño frente a las versiones base y frente a otros métodos como el árbol manual y el bagging. El RandomForest_base supera sin duda al bagging con sklearn, presentando un RMSE cercano a 1786 y un MAE de aproximadamente 1334, gracias a la fusión de muestreo bootstrap y selección aleatoria de variables en cada división, lo que disminuye la correlación entre árboles y optimiza la capacidad de generalización. Al ajustar Random Forest, el error se reduce significativamente (RMSE ≈ 1566, MAE ≈ 1149), lo que demuestra que modificar parámetros como n_estimators, max_depth y max_features facilita alcanzar un mejor balance entre sesgo y varianza: mayor número de árboles y una profundidad óptima contribuyen a captar conexiones no lineales sin incurrir en un sobreajuste excesivo

Por su parte, XGBoost presenta un nivel de error bastante competitivo incluso en su versión base (RMSE ≈ 1563, MAE ≈ 1147), comparable o un poco superior al Random Forest ajustado, dado que el boosting corrige secuencialmente los errores de los árboles anteriores y se centra en las observaciones más desafiantes, creando un modelo más representativo. Luego de la calibración, XGBoost se establece como el modelo de mejor rendimiento global (RMSE ≈ 1548, MAE ≈ 1138), gracias a la optimización de hiperparámetros como learning_rate, gamma y colsample_bytree, que regulan la complejidad, implementan regularización y ajustan el muestreo de columnas, disminuyendo aún más el error sin sacrificar la capacidad de generalización. En contraste, el bagging con sklearn al utilizar max_features = log(n_features) resulta ser el modelo menos efectivo, dado que la fuerte limitación en la cantidad de variables por división genera árboles demasiado débiles (subajuste), y el ensamble retiene esa escasa capacidad predictiva, corroborando que los modelos altamente limitados en información funcionan peor que los ensambles más expresivos y adecuadamente calibrados